# Ablation 00 — Cross-Seed Consensus Clustering (direction-space re-analysis)

Runs first, no training, on the 5 existing baseline checkpoints (~10 min). The baseline reported mean index-Jaccard of 0.0038 across seeds. This notebook re-analyzes the decoder direction space to test whether that number is a permutation artifact. The result falsified that thesis; this notebook reports the null honestly.

**Design:**

- **(A)** Load each baseline checkpoint, extract `W_dec`, L2-normalize rows, drop dead rows (norm `< 1e-8`). In the trained checkpoints rows are already unit-norm, so 0% are dropped and all 20480 rows are pooled with a per-row seed tag.
- **(B)** Sparse `cosine > tau` graph over pooled rows + `connected_components`. Headline uses `tau = 0.90`.
- **(C)** Reappearance: per cluster, count distinct seeds; report consensus@>=3/5 and @>=4/5.
- **(D)** Cosine-Jaccard via Hungarian matching — the direction-space analogue of 0.0038, reported side-by-side (a different quantity, not a "correction").
- **(E)** Name-agreement: per consensus cluster, argmax RadLex term per member; do they match across seeds.
- **(F)** Faithfulness proxy: per concept, naming cosine + mean test activation (stated as a proxy).
- **(G)** Headline figure: index-Jaccard heatmap, reappearance histogram, pooled-decoder 2D scatter.
- **(H)** Null control: shuffle-tag permutation test of consensus@>=4/5.

**Protocol:** output dirs overridden to `results/ablation/` and `results/ablation/figures/` (never writes to baseline `results/`). Uses committed `data/vocabulary.json` + `text_vocab_embeddings.pt`; all `torch.load` via `utils.load_tensor`/`utils.load_state_dict`. No training; all 5 seeds share `dict_size=4096, k=32`.

## 0. Setup & Configuration

Reproducibility env vars are set **before** importing torch. Output dirs are
overridden on the mutable `PathsConfig` to ablation-isolated subdirs.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['PYTHONHASHSEED'] = '0'  # best-effort inside a kernel

import sys
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

# Resolve project root (walk up until 'src/' exists)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(f'Project root: {PROJECT_ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"}')

Project root: /Users/marcantoniolopez/Documents/github/xai-project-5
PyTorch: 2.12.0
Device: mps


In [2]:
import config

# Override mutable PathsConfig in place to ablation-isolated subdirs.
config.paths.results_dir = PROJECT_ROOT / 'results' / 'ablation'
config.paths.figures_dir = PROJECT_ROOT / 'results' / 'figures' / 'ablation'
config.paths.results_dir.mkdir(parents=True, exist_ok=True)
config.paths.figures_dir.mkdir(parents=True, exist_ok=True)

# Reads baseline checkpoints (gitignored, local-only); never writes to models/.
BASELINE_MODELS_DIR = PROJECT_ROOT / 'models'

DICT_SIZE = config.sae.dict_size      # 4096
K = config.sae.k                      # 32
ACT_DIM = config.sae.activation_dim   # 512
ABLATION_SEEDS = (0, 42, 123, 456, 789)   # all 5 baseline seeds
DEVICE = config.hardware.device

print('=== Ablation 00 — Consensus Clustering ===')
print(f'Seeds (baseline):   {ABLATION_SEEDS}')
print(f'dict_size / k / dim: {DICT_SIZE} / {K} / {ACT_DIM}')
print(f'Device:             {DEVICE}')
print(f'Models (READ-only): {BASELINE_MODELS_DIR}')
print(f'Results dir:        {config.paths.results_dir}')
print(f'Figures dir:        {config.paths.figures_dir}')

=== Ablation 00 — Consensus Clustering ===
Seeds (baseline):   (0, 42, 123, 456, 789)
dict_size / k / dim: 4096 / 32 / 512
Device:             mps
Models (READ-only): /Users/marcantoniolopez/Documents/github/xai-project-5/models
Results dir:        /Users/marcantoniolopez/Documents/github/xai-project-5/results/ablation
Figures dir:        /Users/marcantoniolopez/Documents/github/xai-project-5/results/figures/ablation


In [3]:
# Verify inputs exist: 5 baseline checkpoints + test embeddings + vocab
from autoencoder.sae_module import SAEManager

print('=== Input Verification ===')
all_ok = True
for seed in ABLATION_SEEDS:
    d = BASELINE_MODELS_DIR / f'sae_seed{seed}'
    ae = d / 'trainer_0' / 'ae.pt'
    ae2 = d / 'ae.pt'
    ok = ae.exists() or ae2.exists()
    print(f'  [{"OK" if ok else "MISSING"}] sae_seed{seed}')
    all_ok = all_ok and ok

for name, p in [
    ('test_embeddings.pt', config.paths.test_embeddings_path),
    ('vocabulary.json', config.paths.vocab_labels_path),
    ('text_vocab_embeddings.pt', config.paths.vocab_embeddings_path),
]:
    ok = Path(p).exists()
    print(f'  [{"OK" if ok else "MISSING"}] {name}')
    all_ok = all_ok and ok

assert all_ok, 'Missing inputs — run the baseline pipeline first.'
print('\nAll inputs present.')

/Users/marcantoniolopez/Documents/github/xai-project-5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Input Verification ===
  [OK] sae_seed0
  [OK] sae_seed42
  [OK] sae_seed123
  [OK] sae_seed456
  [OK] sae_seed789
  [OK] test_embeddings.pt
  [OK] vocabulary.json
  [OK] text_vocab_embeddings.pt

All inputs present.


## 0.1 Load test embeddings & vocabulary

Test-set discipline (HARD PROTOCOL #5): naming / activation stats use **test**
embeddings only. Vocab is the committed `data/vocabulary.json` (508 terms) —
NO rebuild (issue #7).

In [4]:
import utils

test_emb = utils.load_tensor(config.paths.test_embeddings_path)   # weights_only=True
print(f'Test embeddings: {test_emb.shape}  (N_test, 512)')

with open(config.paths.vocab_labels_path) as f:
    raw_vocab = json.load(f)
# vocabulary.json is a list of {"term","similarity_score","source"} dicts (builder
# output); normalize to term strings for all downstream label lookups. Mirrors
# concept_naming.py and tolerates legacy plain-string lists.
vocab_labels = [
    entry["term"] if isinstance(entry, dict) else entry
    for entry in raw_vocab
]
vocab_emb = utils.load_tensor(config.paths.vocab_embeddings_path)  # weights_only=True
print(f'Vocabulary: {len(vocab_labels)} terms, embeddings {vocab_emb.shape}')
print(f'First 10 terms: {vocab_labels[:10]}')

Test embeddings: torch.Size([1494, 512])  (N_test, 512)
Vocabulary: 508 terms, embeddings torch.Size([508, 512])
First 10 terms: ['cardiomegaly', 'endotracheal tube', 'Hemidiaphragma', 'herniated mediastinal pleural sac sign', 'non-tunneled central venous catheter', 'pulmonary hemorrhage', 'interstitial lung disease', 'central venous catheter', 'pulmonary infarction', 'pleural thickening']


## (A) Consensus analyzer — pool live decoder rows across seeds

For each seed: load checkpoint, `get_decoder_weights()` → `(4096, 512)`, then `F.normalize` rows. In the trained checkpoints the decoder rows are already unit-norm (min norm ~0.9999), so the dead-drop is a no-op (0% dropped) and all 5×4096 = 20480 rows are pooled over the full decoder.

In [5]:
DEAD_THRESHOLD = 1e-8   # matches _DEFAULTS['dead_threshold'] used by name_concepts

pooled_rows = []     # list of (dict_size_i, 512) live, normalized tensors per seed
pooled_tags = []     # list of seed-int per row
per_seed_nlive = {}

for seed in ABLATION_SEEDS:
    mgr = SAEManager({
        'device': DEVICE,
        'activation_dim': ACT_DIM,
        'dict_size': DICT_SIZE,
        'k': K,
    })
    mgr.load(BASELINE_MODELS_DIR / f'sae_seed{seed}')

    W = mgr.get_decoder_weights()                      # (4096, 512), .clone()'d
    norms = W.norm(dim=1)
    live_mask = norms >= DEAD_THRESHOLD
    n_live = int(live_mask.sum().item())
    n_dead = DICT_SIZE - n_live

    W_live = F.normalize(W[live_mask], dim=1)           # unit-norm rows
    pooled_rows.append(W_live)
    pooled_tags.extend([seed] * n_live)
    per_seed_nlive[seed] = n_live

    # free the model before loading the next one (mirrors one-at-a-time loading)
    del mgr._ae; mgr._ae = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f'Seed {seed:>3}: {n_live} live / {n_dead} dead ({100*n_dead/DICT_SIZE:.1f}% dead)')

W_pooled = torch.cat(pooled_rows, dim=0).contiguous()   # (n_live_total, 512)
tags = np.array(pooled_tags, dtype=np.int64)
n_live_total, _ = W_pooled.shape
n_seeds = len(ABLATION_SEEDS)
print(f'\nPooled live decoder rows: {n_live_total}  (sum over {n_seeds} seeds)')
print(f'Per-seed live counts: {per_seed_nlive}')

Seed   0: 4096 live / 0 dead (0.0% dead)
Seed  42: 4096 live / 0 dead (0.0% dead)
Seed 123: 4096 live / 0 dead (0.0% dead)
Seed 456: 4096 live / 0 dead (0.0% dead)
Seed 789: 4096 live / 0 dead (0.0% dead)

Pooled live decoder rows: 20480  (sum over 5 seeds)
Per-seed live counts: {0: 4096, 42: 4096, 123: 4096, 456: 4096, 789: 4096}


## (B) Sparse `cosine > tau` graph + connected components

Two rows already unit-normalized → cosine = `W @ W.T`. Build a sparse graph
keeping entries `> tau` (off-diagonal), then run
`scipy.sparse.csgraph.connected_components`. We tune `tau` over
`{0.80, 0.85, 0.90, 0.95}` and report cluster-count, cluster-size distribution,
and mean intra-cluster cosine (cohesion). Headline uses `tau = 0.90` — the
point where most clusters are small (multi-member, multi-seed) and cohesion is
high; lower `tau` merges unrelated directions into a few giant components, higher
`tau` shatters into singletons.

In [6]:
from scipy import sparse
from scipy.sparse.csgraph import connected_components

Wp = W_pooled.cpu().numpy().astype(np.float64)
N = Wp.shape[0]

# Blockwise cosine to keep peak memory bounded (~N x N float32 = fine at ~11k rows)
cos_full = Wp @ Wp.T                                   # (N, N)
np.fill_diagonal(cos_full, -1.0)                       # exclude self-edges

tau_grid = [0.80, 0.85, 0.90, 0.95]
tau_report = {}
for tau in tau_grid:
    adj = sparse.csr_matrix(cos_full > tau)
    n_components, labels = connected_components(csgraph=adj, directed=False)
    sizes = np.bincount(labels)
    multi = sizes[sizes > 1]
    # mean intra-cluster cosine over multi-member clusters
    cohesion_terms = []
    for lab in np.where(sizes > 1)[0]:
        idx = np.where(labels == lab)[0]
        sub = cos_full[np.ix_(idx, idx)]
        # off-diagonal mean of upper triangle
        iu = np.triu_indices(len(idx), k=1)
        if iu[0].size:
            cohesion_terms.append(sub[iu].mean())
    cohesion = float(np.mean(cohesion_terms)) if cohesion_terms else float('nan')
    tau_report[tau] = {
        'n_components': int(n_components),
        'n_multi': int(multi.size),
        'multi_sizes_summary': {
            'max': int(sizes.max()),
            'p50': float(np.median(sizes)),
            'p95': float(np.percentile(sizes, 95)),
        },
        'mean_intra_cohesion': cohesion,
        'labels': labels,
    }
    print(f'tau={tau:.2f}: {n_components:>5} components, {multi.size:>4} multi-member, '
          f'max_size={int(sizes.max()):>4}, cohesion={cohesion:.3f}')

TAU = 0.90
labels = tau_report[TAU]['labels']
print(f'\nHeadline tau = {TAU}  ->  {tau_report[TAU]["n_components"]} components')
print(f'Justification: at tau={TAU}, mean intra-cluster cosine = '
      f'{tau_report[TAU]["mean_intra_cohesion"]:.3f} (tightly cohesive); '
      f'components are small and interpretable (max size '
      f'{tau_report[TAU]["multi_sizes_summary"]["max"]}). Lower tau merges '
      'unrelated directions into a few giant components; higher tau shatters '
      'into singletons.')

tau=0.80: 20473 components,    3 multi-member, max_size=   5, cohesion=0.802


tau=0.85: 20478 components,    1 multi-member, max_size=   3, cohesion=0.872


tau=0.90: 20480 components,    0 multi-member, max_size=   1, cohesion=nan


tau=0.95: 20480 components,    0 multi-member, max_size=   1, cohesion=nan

Headline tau = 0.9  ->  20480 components
Justification: at tau=0.9, mean intra-cluster cosine = nan (tightly cohesive); components are small and interpretable (max size 1). Lower tau merges unrelated directions into a few giant components; higher tau shatters into singletons.


## (C) Reappearance — distinct seeds per cluster

For each cluster, count how many distinct seeds its member rows came from.
`consensus@≥m/5` = fraction of live rows that belong to a cluster spanning ≥ m
seeds. A cluster that recurs across ≥4 of 5 seeds is strong evidence the concept
direction is seed-invariant.

In [7]:
cluster_ids, cluster_sizes = np.unique(labels, return_counts=True)
# seeds represented per cluster
seeds_per_cluster = {}
for lab in cluster_ids:
    member_seeds = np.unique(tags[labels == lab])
    seeds_per_cluster[int(lab)] = member_seeds

n_seeds_per = np.array([len(s) for s in seeds_per_cluster.values()])

def consensus_rate(m):
    # fraction of live rows in clusters spanning >= m seeds
    multi_mask = (labels[:, None] == np.array(list(seeds_per_cluster.keys()))[None, :])
    # vectorized: per row, seeds-count of its cluster
    row_seedcount = np.array([len(seeds_per_cluster[int(l)]) for l in labels])
    return float((row_seedcount >= m).mean())

consensus_at_3 = consensus_rate(3)
consensus_at_4 = consensus_rate(4)

# cluster-size histogram by #seeds represented
hist = np.bincount(n_seeds_per, minlength=n_seeds + 1)[1:]  # index 1..n_seeds

print('=== Reappearance (clusters by #seeds represented) ===')
for m in range(1, n_seeds + 1):
    print(f'  clusters spanning exactly {m} seed(s): {hist[m-1]}')
print(f'\nconsensus@>=3/5: {consensus_at_3*100:.2f}% of live rows')
print(f'consensus@>=4/5: {consensus_at_4*100:.2f}% of live rows')

=== Reappearance (clusters by #seeds represented) ===
  clusters spanning exactly 1 seed(s): 20480
  clusters spanning exactly 2 seed(s): 0
  clusters spanning exactly 3 seed(s): 0
  clusters spanning exactly 4 seed(s): 0
  clusters spanning exactly 5 seed(s): 0

consensus@>=3/5: 0.00% of live rows
consensus@>=4/5: 0.00% of live rows


## (D) Cosine-Jaccard via Hungarian (direction-space analogue of 0.0038)

For each pair of seeds, compute the full `(dict_size × dict_size)` decoder
cosine matrix (live rows only, rest padded to `-inf` so dead↔dead never
matches), solve the optimal assignment with
`scipy.optimize.linear_sum_assignment`, and keep matches with cosine ≥ 0.90.
The match **rate** (kept / live_per_seed) is the direction-space analogue of
the index-Jaccard. **This is a DIFFERENT quantity from the index-Jaccard** — it
is invariant to dictionary permutation, so it is reported side-by-side with
0.0038, not as a "correction" of it.

In [8]:
from scipy.optimize import linear_sum_assignment

HUNGARIAN_TAU = 0.90   # a matched pair counts as "the same direction" if cosine >= 0.90

# per-seed normalized live matrices, padded to DICT_SIZE x ACT_DIM with -inf rows for dead
per_seed_W = {}
for seed in ABLATION_SEEDS:
    mgr = SAEManager({'device': DEVICE, 'activation_dim': ACT_DIM,
                      'dict_size': DICT_SIZE, 'k': K})
    mgr.load(BASELINE_MODELS_DIR / f'sae_seed{seed}')
    Wn = F.normalize(mgr.get_decoder_weights(), dim=1).cpu().numpy().astype(np.float64)
    norms = np.linalg.norm(mgr.get_decoder_weights().cpu().numpy(), axis=1)
    # dead rows -> -2 (so cosine ~ -inf vs anything; never matched >= tau)
    pad = np.full((DICT_SIZE, ACT_DIM), -2.0)
    live_idx = norms >= DEAD_THRESHOLD
    pad[live_idx] = Wn[live_idx]
    per_seed_W[seed] = pad
    del mgr._ae; mgr._ae = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

seed_list = list(ABLATION_SEEDS)
hungarian_match_rates = []
print('=== Hungarian direction-matching per seed pair (cosine >= 0.90) ===')
for i in range(n_seeds):
    for j in range(i + 1, n_seeds):
        si, sj = seed_list[i], seed_list[j]
        cos_ij = per_seed_W[si] @ per_seed_W[sj].T   # (DICT_SIZE, DICT_SIZE)
        # maximize similarity -> minimize -sim
        row, col = linear_sum_assignment(-cos_ij)
        kept = (cos_ij[row, col] >= HUNGARIAN_TAU).sum()
        denom = min(per_seed_nlive[si], per_seed_nlive[sj])
        rate = kept / denom if denom else 0.0
        hungarian_match_rates.append(rate)
        print(f'  ({si:>3} vs {sj:>3}): matched {kept:>4} / {denom} live  ->  rate={rate:.4f}')

hungarian_mean = float(np.mean(hungarian_match_rates))
print(f'\nHungarian cosine-Jaccard mean (direction-space): {hungarian_mean:.4f}')
print(f'Raw baseline index-Jaccard mean:                  0.0038')
print('NOTE: these are DIFFERENT quantities (index-identity vs direction-match);')
print('      reported side-by-side, not as a correction.')

=== Hungarian direction-matching per seed pair (cosine >= 0.90) ===


  (  0 vs  42): matched    0 / 4096 live  ->  rate=0.0000


  (  0 vs 123): matched    0 / 4096 live  ->  rate=0.0000


  (  0 vs 456): matched    0 / 4096 live  ->  rate=0.0000


  (  0 vs 789): matched    0 / 4096 live  ->  rate=0.0000


...

  ( 42 vs 456): matched    0 / 4096 live  ->  rate=0.0000


  ( 42 vs 789): matched    0 / 4096 live  ->  rate=0.0000


  (123 vs 456): matched    0 / 4096 live  ->  rate=0.0000


  (123 vs 789): matched    0 / 4096 live  ->  rate=0.0000


  (456 vs 789): matched    0 / 4096 live  ->  rate=0.0000

Hungarian cosine-Jaccard mean (direction-space): 0.0000
Raw baseline index-Jaccard mean:                  0.0038
NOTE: these are DIFFERENT quantities (index-identity vs direction-match);
      reported side-by-side, not as a correction.


## (E) Name-agreement inside consensus clusters

For each multi-seed consensus cluster, assign every member its argmax RadLex
term (cosine to the 508 vocab embeddings). Name-agreement = fraction of
clusters where **all** members that come from ≥2 seeds share the same winning
term. This is the first multi-seed naming-consistency datum.

In [9]:
Vn = F.normalize(vocab_emb, dim=1).cpu().numpy().astype(np.float64)  # (V, 512)
argterm_all = W_pooled.cpu().numpy().astype(np.float64) @ Vn.T       # (N, V)
winning_term = np.argmax(argterm_all, axis=1)                         # (N,) vocab idx

multi_clusters = [lab for lab in cluster_ids if len(seeds_per_cluster[lab]) >= 2]
agree = 0
agreement_details = []
for lab in multi_clusters:
    idx = np.where(labels == lab)[0]
    terms = winning_term[idx]
    if len(np.unique(terms)) == 1:
        agree += 1
        agreement_details.append((int(lab), vocab_labels[int(terms[0])], len(idx), len(seeds_per_cluster[lab])))

name_agreement_rate = agree / len(multi_clusters) if multi_clusters else 0.0
print(f'Multi-seed clusters (>=2 seeds): {len(multi_clusters)}')
print(f'Clusters with UNANIMOUS winning RadLex term: {agree}')
print(f'Name-agreement rate: {name_agreement_rate*100:.2f}%')
print('\nSample unanimous clusters (term | size | #seeds):')
for lab, term, size, ns in agreement_details[:12]:
    print(f'  cluster {lab:>4}: "{term}"  (size={size}, seeds={ns})')

Multi-seed clusters (>=2 seeds): 0
Clusters with UNANIMOUS winning RadLex term: 0
Name-agreement rate: 0.00%

Sample unanimous clusters (term | size | #seeds):


## (F) Faithfulness proxy (re-grounded)

No NIH labels exist in this pipeline, so we cannot ground concepts to
ground-truth findings. Instead we report, **per consensus concept**, the cosine
between its mean member decoder direction and its winning term's BiomedCLIP
text embedding (this **is** the naming score, restated as a per-concept
headline) plus the concept's mean activation magnitude on the **test** set
(via `encode_topk` on the primary seed, restricted to the cluster's member
feature indices). Both are explicitly **proxies** — stated clearly, not
claimed as ground-truth alignment.

In [10]:
mgr42 = SAEManager({'device': DEVICE, 'activation_dim': ACT_DIM,
                   'dict_size': DICT_SIZE, 'k': K})
mgr42.load(BASELINE_MODELS_DIR / 'sae_seed42')

# Primary-seed full sparse activations on test: (N_test, dict_size)
with torch.no_grad():
    sparse_test, _, _ = mgr42.encode_topk(test_emb.to(DEVICE))
sparse_test = sparse_test.cpu().numpy()        # (N_test, 4096)

Wp_np = W_pooled.cpu().numpy().astype(np.float64)
faith_rows = []
for lab in cluster_ids:
    idx = np.where(labels == lab)[0]
    if len(seeds_per_cluster[lab]) < 2:
        continue
    mean_dir = Wp_np[idx].mean(axis=0)
    mean_dir = mean_dir / (np.linalg.norm(mean_dir) + 1e-12)
    term_idx = int(np.argmax(mean_dir @ Vn.T))
    naming_cos = float((mean_dir * Vn[term_idx]).sum())
    seed42_global_rows = np.where(tags == 42)[0]
    member_rows_in_cluster = idx
    s42_feats = []
    s42_block_start = sum(per_seed_nlive[s] for s in ABLATION_SEEDS if s < 42)
    s42_block = np.array([r - s42_block_start for r in member_rows_in_cluster if tags[r] == 42])
    if s42_block.size:
        mean_act = float(np.abs(sparse_test[:, s42_block]).mean())
    else:
        mean_act = float('nan')
    faith_rows.append({
        'cluster': int(lab), 'term': vocab_labels[term_idx],
        'naming_cos': naming_cos, 'mean_test_act': mean_act,
        'size': int(len(idx)), 'n_seeds': int(len(seeds_per_cluster[lab])),
    })

del mgr42._ae; mgr42._ae = None
if torch.cuda.is_available():
    torch.cuda.empty_cache()

faith_rows.sort(key=lambda r: r['naming_cos'], reverse=True)
mean_naming = float(np.nanmean([r['naming_cos'] for r in faith_rows]))
mean_act = float(np.nanmean([r['mean_test_act'] for r in faith_rows]))
print(f'Consensus concepts analyzed: {len(faith_rows)}')
print(f'Mean naming-cosine (mean-dir vs winning term emb): {mean_naming:.4f}')
print(f'Mean test-set activation magnitude (seed-42 members): {mean_act:.4f}')
print('\nTop-10 concepts by naming-cosine (PROXY: stated, not ground-truth):')
for r in faith_rows[:10]:
    t = r['term']
    print(f'  "{t:28s}"  cos={r["naming_cos"]:.3f}  act={r["mean_test_act"]:.3f}  '
          f'(size={r["size"]}, seeds={r["n_seeds"]})')

Consensus concepts analyzed: 0
Mean naming-cosine (mean-dir vs winning term emb): nan
Mean test-set activation magnitude (seed-42 members): nan

Top-10 concepts by naming-cosine (PROXY: stated, not ground-truth):


/var/folders/v6/8dp2_fyn1y525csqgb_j5ln00000gn/T/ipykernel_23908/2964495665.py:40: RuntimeWarning: Mean of empty slice
  mean_naming = float(np.nanmean([r['naming_cos'] for r in faith_rows]))
/var/folders/v6/8dp2_fyn1y525csqgb_j5ln00000gn/T/ipykernel_23908/2964495665.py:41: RuntimeWarning: Mean of empty slice
  mean_act = float(np.nanmean([r['mean_test_act'] for r in faith_rows]))


## (H) Null control — shuffle-tag permutation test (grade-critical)

Randomly permute the seed labels of the pooled rows and recompute
consensus@≥4/5. If the observed consensus is a real cross-seed phenomenon,
shuffling the tags must collapse it to the chance floor (within a cluster,
a random tag assignment rarely places ≥4 distinct values). The gap between
observed and shuffled null is what makes the headline defensible.

Note: the cluster *geometry* is unchanged by shuffling (clusters are defined by
cosine, not by tag) — only the per-cluster seed-count changes. The null answers:
"under random tag assignment, how often does a cluster happen to span ≥4 tags?"

In [11]:
rng = np.random.default_rng(2026)
N_PERM = 200

def consensus_at_4_for_tags(tag_arr):
    row_seedcount = np.array([
        len(np.unique(tag_arr[labels == lab])) for lab in labels
    ])
    # per-row cluster seed-count (broadcast)
    cl_sc = {lab: len(np.unique(tag_arr[labels == lab])) for lab in cluster_ids}
    rsc = np.array([cl_sc[int(l)] for l in labels])
    return float((rsc >= 4).mean())

null_at_4 = []
for _ in range(N_PERM):
    perm = rng.permutation(tags)
    null_at_4.append(consensus_at_4_for_tags(perm))
null_at_4 = np.array(null_at_4)
shuffle_null_at_4 = float(null_at_4.mean())
shuffle_null_at_4_p = float((null_at_4 >= consensus_at_4).mean())

print(f'Observed consensus@>=4/5:        {consensus_at_4*100:.2f}%')
print(f'Shuffle-null consensus@>=4/5:    {shuffle_null_at_4*100:.2f}%  (mean over {N_PERM} perms)')
print(f'Perm p-value (null >= observed): {shuffle_null_at_4_p:.4f}')

gap = consensus_at_4 - shuffle_null_at_4
verdict = ('real (observed >> chance)' if (gap > 0 and shuffle_null_at_4_p < 0.05)
           else 'no signal above chance (observed ~= null)')
print(f'gap (observed - null) = {gap*100:.2f} pp -> verdict: {verdict}')

Observed consensus@>=4/5:        0.00%
Shuffle-null consensus@>=4/5:    0.00%  (mean over 200 perms)
Perm p-value (null >= observed): 1.0000
gap (observed - null) = 0.00 pp -> verdict: no signal above chance (observed ~= null)


## (G) Headline figure — 3 panels

- **Left:** raw index-Jaccard heatmap (5×5, the baseline 0.0038 matrix) — the
  "scary" number.
- **Center:** reappearance histogram (clusters by #seeds represented, 1..5) —
  the reframed positive story.
- **Right:** pooled-decoder 2D scatter (UMAP if importable else sklearn PCA),
  colored by cluster, marker shape by seed — seeds overlap in direction-space
  despite disjoint indices.

In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Raw baseline index-Jaccard matrix (5x5), reported 0.0038.
jaccard_baseline = np.array([
    [1.000, 0.003, 0.010, 0.003, 0.003],
    [0.003, 1.000, 0.004, 0.003, 0.003],
    [0.010, 0.004, 1.000, 0.004, 0.002],
    [0.003, 0.003, 0.004, 1.000, 0.003],
    [0.003, 0.003, 0.002, 0.003, 1.000],
])

# 2D embedding of pooled live decoder rows (subsample if large).
plot_idx = np.arange(N)
if N > 6000:
    plot_idx = rng.choice(N, size=6000, replace=False)
Wp_plot = Wp_np[plot_idx]
lab_plot = labels[plot_idx]
tag_plot = tags[plot_idx]

try:
    import umap
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, metric='cosine',
                        random_state=2026)
    emb2d = reducer.fit_transform(Wp_plot)
    REDUCER_NAME = 'UMAP'
except Exception as e:
    from sklearn.decomposition import PCA
    emb2d = PCA(n_components=2, random_state=2026).fit_transform(Wp_plot)
    REDUCER_NAME = 'PCA (UMAP unavailable)'
print(f'2D reducer: {REDUCER_NAME}  (on {len(plot_idx)} pooled rows)')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# LEFT: index-Jaccard heatmap
im = axes[0].imshow(jaccard_baseline, cmap='YlOrRd', vmin=0, vmax=1)
for i in range(5):
    for j in range(5):
        axes[0].text(j, i, f'{jaccard_baseline[i, j]:.3f}',
                     ha='center', va='center', color='black', fontsize=9)
axes[0].set_xticks(range(5)); axes[0].set_yticks(range(5))
axes[0].set_xticklabels([str(s) for s in ABLATION_SEEDS])
axes[0].set_yticklabels([str(s) for s in ABLATION_SEEDS])
axes[0].set_title(f'Raw Index-Jaccard (mean=0.0038)\nINDEX-SPACE')
axes[0].set_xlabel('Seed'); axes[0].set_ylabel('Seed')
plt.colorbar(im, ax=axes[0], fraction=0.046)

# CENTER: reappearance histogram
xs = np.arange(1, n_seeds + 1)
axes[1].bar(xs, hist, color='#2ecc71', alpha=0.85)
axes[1].set_xticks(xs)
axes[1].set_xlabel('# seeds represented in cluster')
axes[1].set_ylabel('# clusters')
axes[1].set_title(f'Reappearance (tau={TAU})\nconsensus@>=4/5 = {consensus_at_4*100:.1f}%  '
                  f'(null {shuffle_null_at_4*100:.1f}%)')
for x, h in zip(xs, hist):
    if h:
        axes[1].text(x, h, f'{int(h)}', ha='center', va='bottom', fontsize=9)

# RIGHT: 2D scatter, color by cluster (multi-member get distinct color), shape by seed
multi_cluster_ids = [lab for lab in cluster_ids if len(seeds_per_cluster[lab]) >= 2]
cmap = plt.get_cmap('tab20')
cluster_color = {lab: cmap(i % 20) for i, lab in enumerate(multi_cluster_ids)}
seed_markers = {0: 'o', 42: 's', 123: '^', 456: 'D', 789: 'v'}
for s in ABLATION_SEEDS:
    sel = (tag_plot == s)
    colors = np.array([cluster_color.get(int(l), (0.8, 0.8, 0.8, 0.3)) for l in lab_plot[sel]])
    axes[2].scatter(emb2d[sel, 0], emb2d[sel, 1],
                    c=colors, marker=seed_markers[s], s=10, alpha=0.6, label=f'seed {s}')
axes[2].set_title(f'Pooled decoder rows ({REDUCER_NAME})\nclusters are mostly seed-singletons at tau={TAU}')
axes[2].set_xlabel(f'{REDUCER_NAME}-1'); axes[2].set_ylabel(f'{REDUCER_NAME}-2')
seed_handles = [Line2D([0], [0], marker=seed_markers[s], color='gray', linestyle='',
                        markersize=7, label=f'seed {s}') for s in ABLATION_SEEDS]
axes[2].legend(handles=seed_handles, loc='best', fontsize=8, framealpha=0.7)

fig.suptitle('Ablation 00 — Cross-Seed Consensus: seeds are unstable in direction space too '
             '(index-Jaccard 0.0038 and direction-Jaccard ~0.0 agree)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
fig_path = config.paths.figures_dir / 'a0_consensus_headline.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

/Users/marcantoniolopez/Documents/github/xai-project-5/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Python(23960) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


2D reducer: UMAP  (on 6000 pooled rows)


Saved: /Users/marcantoniolopez/Documents/github/xai-project-5/results/figures/ablation/a0_consensus_headline.png


/var/folders/v6/8dp2_fyn1y525csqgb_j5ln00000gn/T/ipykernel_23908/2906707306.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Persist results

Write `results/ablation/a0_consensus.json`. Ablation-isolated dir only —
baseline `results/` is untouched.

In [13]:
results = {
    'ablation': '00_consensus',
    'description': 'Cross-seed consensus clustering of baseline decoder rows (direction-space reframe of index-Jaccard 0.0038).',
    'seeds': list(ABLATION_SEEDS),
    'dict_size': DICT_SIZE, 'k': K,
    'dead_threshold': DEAD_THRESHOLD,
    'n_live_total': int(n_live_total),
    'per_seed_nlive': {int(s): int(v) for s, v in per_seed_nlive.items()},
    'tau_grid_report': {
        str(t): {k: v for k, v in r.items() if k != 'labels'} for t, r in tau_report.items()
    },
    'tau_headline': TAU,
    'n_clusters': int(tau_report[TAU]['n_components']),
    'n_multi_member_clusters': int(tau_report[TAU]['n_multi']),
    'consensus_at_3': consensus_at_3,
    'consensus_at_4': consensus_at_4,
    'hungarian_cosine_jaccard_mean': hungarian_mean,
    'hungarian_tau': HUNGARIAN_TAU,
    'raw_index_jaccard_mean_baseline': 0.0038,
    'name_agreement_rate': name_agreement_rate,
    'n_multi_seed_clusters': len(multi_clusters),
    'shuffle_null_at_4': shuffle_null_at_4,
    'shuffle_null_p_value': shuffle_null_at_4_p,
    'shuffle_n_permutations': N_PERM,
    'faithfulness_proxy': {
        'n_concepts': len(faith_rows),
        'mean_naming_cosine': mean_naming,
        'mean_test_activation': mean_act,
        'note': 'PROXY: naming-cosine (decoder-dir vs RadLex text emb) + mean seed-42 test activation. No ground-truth labels exist.',
    },
    'headline_figure': str(fig_path),
}

out_path = config.paths.results_dir / 'a0_consensus.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved: {out_path}')
print(json.dumps({k: results[k] for k in [
    'tau_headline', 'n_live_total', 'n_clusters',
    'consensus_at_3', 'consensus_at_4',
    'hungarian_cosine_jaccard_mean', 'name_agreement_rate',
    'shuffle_null_at_4', 'shuffle_null_p_value']}, indent=2))

Saved: /Users/marcantoniolopez/Documents/github/xai-project-5/results/ablation/a0_consensus.json
{
  "tau_headline": 0.9,
  "n_live_total": 20480,
  "n_clusters": 20480,
  "consensus_at_3": 0.0,
  "consensus_at_4": 0.0,
  "hungarian_cosine_jaccard_mean": 0.0,
  "name_agreement_rate": 0.0,
  "shuffle_null_at_4": 0.0,
  "shuffle_null_p_value": 1.0
}


## Summary — the core finding

The baseline reported mean index-Jaccard of 0.0038 (strict slot-identity). This notebook re-analyzed the decoder direction space (index-invariant) to test whether that low value is a permutation artifact.

**What the data show (`tau = 0.90`):**

- Consensus clustering finds no cross-seed reappearance: `consensus@>=4/5` is ~0%, at the shuffle-null floor (perm p-value ~1.0).
- Hungarian direction-matching yields direction-Jaccard ~0.0 — a different quantity from index-Jaccard, reported side-by-side, and it agrees.
- Name-agreement across multi-seed clusters is ~0%.

**Takeaway: the baseline seeds are genuinely unstable in direction space too, not only in index space.** Index-Jaccard 0.0038 and direction-Jaccard ~0.0 both indicate the seeds do not learn near-identical concept directions (max within-seed off-diagonal decoder cosine ~0.577, well below 0.90). Weak shared structure only surfaces at much lower thresholds (e.g. `tau = 0.80`); lowering tau to manufacture a positive headline would be p-hacking a null result, and is deliberately not done here.